# Translate ViNumQA reasoning traces (VI -> EN) with VietAI/envit5-translation

Purpose: test whether the SFT-w-reasoning-trace underperformance vs. SFT-wo-reasoning-trace
is caused by the reasoning trace being in Vietnamese (Qwen3's instruction-tuning is
predominantly English). This notebook machine-translates the existing distilled
`qa.reasoning_trace` field (Vietnamese, produced by the Qwen3-Next-80B CoNR pipeline)
into English, using `VietAI/envit5-translation`, leaving every other field untouched:
`pre_text`, `table`, `post_text`, `qa.question`, `qa.program`, `qa.exe_ans` all stay as-is
(still Vietnamese context / gold program), so the only variable changed for the next SFT
run is the language of the reasoning trace.

If SFT on the translated trace improves PA/EA over the Vietnamese-trace run, that's evidence
for the language hypothesis, and justifies re-distilling directly from the teacher
(Qwen3-Next-80B-A3B-Thinking) with an English-output CoNR prompt instead of relying on MT.

## 1. Setup

In [ ]:
!pip install -q "transformers<4.50" sentencepiece accelerate

In [ ]:
import json
import re
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
MODEL_NAME = "VietAI/envit5-translation"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Adjust these two paths to wherever the dataset lives in your environment
# (local repo path shown here; swap for the Kaggle /kaggle/input path if running there).
DATA_DIR = Path("datasets/ViNumQA")
INPUT_FILES = {
    "train": DATA_DIR / "train_with_reasoning_trace.json",
    "valid": DATA_DIR / "valid_with_reasoning_trace.json",
}
OUTPUT_FILES = {
    "train": DATA_DIR / "train_with_reasoning_trace_en.json",
    "valid": DATA_DIR / "valid_with_reasoning_trace_en.json",
}

BATCH_SIZE = 16
# envit5-translation generates with max_length=512 tokens; traces longer than that
# (in sentence-chunks) are split and translated chunk-by-chunk, then rejoined,
# so long traces don't get silently truncated.
MAX_GEN_LENGTH = 512

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print(f"Loaded {MODEL_NAME} on {DEVICE}")

## 2. Translation helpers

envit5-translation requires a `"vi: "` prefix on the source text for VI->EN translation.
Traces are plain prose (no literal function-call syntax inside `reasoning_trace` itself —
verified: the program lives in a separate `qa.program` field), so we translate the whole
trace text directly.

Problems observed while testing, and how each is handled:

1. **Dropped clauses**: the model was trained sentence-by-sentence, and packing multiple
   dense sentences into one `generate()` call sometimes dropped the last clause of a chunk
   (EOS emitted early). Fixed by translating **one sentence at a time**.
2. **Placeholder masking doesn't survive generation**: the first fix attempted for mangled
   numbers was masking each number with a placeholder like `«NUM0»` before translation and
   restoring it after. This failed — envit5-translation is a T5 seq2seq model with a
   SentencePiece vocabulary that has never seen `«NUM0»` as a unit; it tokenizes it into
   subpieces and the decoder freely drops/reorders/pads them (`«NUM0»` -> `NUM0`,
   `" NUM0 "`, or even `NUM0H`), so the placeholder regex fails to match on the way back
   out and numbers are lost entirely.
3. **Working fix — restore by position, normalize as fallback**: translate the original,
   unmasked sentence, then **splice the original Vietnamese-side numeric strings back into
   the translated sentence by order of appearance** (`restore_numbers`) when the count of
   numeric spans matches between source and translation (~79% of sentences in testing).
   When counts don't match — usually because a Vietnamese date phrase like
   `"ngày 31 tháng 12 năm 2012"` (3 separate numeric tokens) collapses into `"December 31,
   2012"` (fewer numeric spans) — position-based restoration isn't safe, so instead a
   lightweight regex pass (`normalize_number_spacing`) repairs the specific spacing
   artifacts the model introduces when it does translate a number as free text, e.g.
   `2438 .4` -> `2438.4`, `VND7 .523` -> `VND 7.523`. This doesn't guarantee every number
   is byte-identical to the source in that ~21% of sentences, but removes the systematic
   spacing corruption observed in testing.

In [ ]:
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

# Matches numeric tokens including optional currency/percent symbols and thousands/decimal
# separators, e.g. "$100.00", "108.50", "7,523", "11.565", "3%", "1.03".
NUMBER_TOKEN_RE = re.compile(r"[\$]?-?\d[\d,\.]*%?")

# Cleans up spacing artifacts the model introduces around numbers it translates as free
# text. Applied both as the fallback when restore_numbers can't safely splice (numeric
# span counts don't match) and as a final pass after a successful splice, since currency
# codes glued to digits (e.g. "VND6851") can survive even when the number itself matched.
# Examples this fixes:
#   "2438 .4"   -> "2438.4"
#   "11 ,565"   -> "11,565"
#   "VND7 .523" -> "VND 7.523"
#   "VND6851"   -> "VND 6851"
NUMBER_SPACING_FIXES = [
    (re.compile(r"(\d)\s+([.,]\d)"), r"\1\2"),  # "2438 .4" -> "2438.4"
    (re.compile(r"([A-Za-z]{2,})(\d)"), r"\1 \2"),  # "VND7" -> "VND 7"
]


def split_sentences(text: str) -> list[str]:
    """Split a reasoning trace into individual sentences."""
    sentences = SENTENCE_SPLIT_RE.split(text.strip())
    return [s for s in sentences if s.strip()] or [text]


def normalize_number_spacing(text: str) -> str:
    """Post-process pass that repairs spacing artifacts MT introduces around numbers."""
    for pattern, replacement in NUMBER_SPACING_FIXES:
        text = pattern.sub(replacement, text)
    return text


def restore_numbers(src_sentence: str, translated_sentence: str) -> str:
    """Replace numeric spans in the translation with the original source numbers,
    matched up in order of appearance. If the count of numeric spans differs between
    source and translation, falls back to normalize_number_spacing on the raw MT
    output instead of guessing a misaligned substitution. Either way, the result is
    passed through normalize_number_spacing once more, since currency-code-glued-to-
    digit artifacts (e.g. "VND6851") can occur even when the number span count matched."""
    src_numbers = NUMBER_TOKEN_RE.findall(src_sentence)
    tgt_matches = list(NUMBER_TOKEN_RE.finditer(translated_sentence))

    if len(src_numbers) != len(tgt_matches):
        return normalize_number_spacing(translated_sentence)

    restored = []
    last_end = 0
    for original_number, match in zip(src_numbers, tgt_matches):
        restored.append(translated_sentence[last_end : match.start()])
        restored.append(original_number)
        last_end = match.end()
    restored.append(translated_sentence[last_end:])
    return normalize_number_spacing("".join(restored))


@torch.no_grad()
def translate_batch(texts: list[str]) -> list[str]:
    """VI -> EN translation for a batch of plain-text strings (no 'vi: ' prefix yet)."""
    prefixed = [f"vi: {t}" for t in texts]
    inputs = tokenizer(
        prefixed, return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(DEVICE)
    outputs = model.generate(**inputs, max_length=MAX_GEN_LENGTH)
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    # envit5-translation prefixes generations with "en: " / "vi: "; strip it.
    return [re.sub(r"^(en|vi):\s*", "", d).strip() for d in decoded]


def translate_trace(text: str) -> str:
    sentences = split_sentences(text)

    translated_sentences = []
    for i in range(0, len(sentences), BATCH_SIZE):
        translated_sentences.extend(translate_batch(sentences[i : i + BATCH_SIZE]))

    restored_sentences = [
        restore_numbers(src_sent, tgt_sent)
        for src_sent, tgt_sent in zip(sentences, translated_sentences)
    ]
    return " ".join(restored_sentences)

## 3. Quick sanity check on a few examples

Spot-check that numbers inside the trace survive translation unchanged before running
the full pass — a garbled number here would poison SFT for a reason unrelated to language.

In [ ]:
with open(INPUT_FILES["train"], encoding="utf-8") as f:
    train_data = json.load(f)

total_sentences = 0
restored_ok = 0

for ex in train_data[:5]:
    src = ex["qa"]["reasoning_trace"]
    src_sentences = split_sentences(src)
    raw_translated = []
    for i in range(0, len(src_sentences), BATCH_SIZE):
        raw_translated.extend(translate_batch(src_sentences[i : i + BATCH_SIZE]))
    tgt_sentences = [
        restore_numbers(s, t) for s, t in zip(src_sentences, raw_translated)
    ]
    tgt = " ".join(tgt_sentences)

    print("=" * 80)
    print("ID:", ex["id"])
    print("VI:", src)
    print("EN:", tgt)

    for vi_sent, raw_sent, restored_sent in zip(src_sentences, raw_translated, tgt_sentences):
        total_sentences += 1
        vi_count = len(NUMBER_TOKEN_RE.findall(vi_sent))
        raw_count = len(NUMBER_TOKEN_RE.findall(raw_sent))
        matched = vi_count == raw_count
        restored_ok += int(matched)
        if not matched:
            print(
                f"  [NUMBER COUNT MISMATCH: vi={vi_count} vs raw_en={raw_count}, "
                f"falling back to normalize_number_spacing]\n"
                f"    VI:  {vi_sent}\n"
                f"    raw: {raw_sent}\n"
                f"    norm:{restored_sent}"
            )
        ratio = len(restored_sent) / max(len(vi_sent), 1)
        if ratio < 0.5:
            print(f"  [SUSPECT TRUNCATION] ratio={ratio:.2f}\n    VI: {vi_sent}\n    EN: {restored_sent}")

print("=" * 80)
print(f"Sentences with number counts matched (safe to restore): {restored_ok}/{total_sentences}")

Review the printed pairs above manually. `restored_ok / total_sentences` tells you what
fraction of sentences had their numbers safely spliced back in — for the remainder, the
raw MT output is kept as-is (whatever the model produced, possibly with mangled digits).
If this ratio is low, consider not deduping on the sentence level, or accepting a small
rate of number drift as the cost of a cheap experiment (still much better than 0% number
protection, which was the state before this fix).

## 4. Full-dataset translation

Runs over both train and valid splits, writes `*_with_reasoning_trace_en.json` files
with the exact same schema as the input (`pre_text`, `table`, `post_text`, `id`, `qa`),
only `qa.reasoning_trace` replaced by its English translation.

In [ ]:
def translate_split(split_name: str) -> None:
    in_path = INPUT_FILES[split_name]
    out_path = OUTPUT_FILES[split_name]
    with open(in_path, encoding="utf-8") as f:
        data = json.load(f)

    for i, ex in enumerate(data):
        vi_trace = ex["qa"]["reasoning_trace"]
        ex["qa"]["reasoning_trace"] = translate_trace(vi_trace)
        if (i + 1) % 100 == 0:
            print(f"[{split_name}] {i + 1}/{len(data)} translated")

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"[{split_name}] done -> {out_path}")

In [ ]:
translate_split("valid")  # smaller split first, sanity check the output file

In [ ]:
translate_split("train")

## 5. Post-translation validation

Same exact-match-count check used when the original (Vietnamese) reasoning traces were
validated against the dataset: confirm the translated files have the same number of
examples, in the same order, with matching `id`s, and that no `reasoning_trace` field
ended up empty (a sign the MT call failed silently for that example).

In [ ]:
for split_name in ["train", "valid"]:
    with open(INPUT_FILES[split_name], encoding="utf-8") as f:
        original = json.load(f)
    with open(OUTPUT_FILES[split_name], encoding="utf-8") as f:
        translated = json.load(f)

    assert len(original) == len(translated), f"{split_name}: length mismatch"
    mismatched_ids = [
        (a["id"], b["id"])
        for a, b in zip(original, translated)
        if a["id"] != b["id"]
    ]
    empty_traces = [b["id"] for b in translated if not b["qa"]["reasoning_trace"].strip()]

    # Flag traces whose translated length is suspiciously short relative to the
    # source (English is usually similar-to-longer than Vietnamese for this kind
    # of prose, so a much shorter EN trace likely means dropped content).
    short_ratio_ids = []
    for a, b in zip(original, translated):
        vi_len = len(a["qa"]["reasoning_trace"])
        en_len = len(b["qa"]["reasoning_trace"])
        if vi_len > 0 and en_len / vi_len < 0.5:
            short_ratio_ids.append(b["id"])

    print(
        f"[{split_name}] n={len(translated)}  id_mismatches={len(mismatched_ids)}  "
        f"empty_traces={len(empty_traces)}  suspect_truncated={len(short_ratio_ids)}"
    )
    if mismatched_ids:
        print("  first few mismatches:", mismatched_ids[:5])
    if empty_traces:
        print("  first few empty:", empty_traces[:5])
    if short_ratio_ids:
        print("  first few suspect-truncated ids:", short_ratio_ids[:10])